# 03 - DML, Histórico e Time Travel (Delta Lake)

Executa operações DML (**INSERT**, **UPDATE**, **DELETE**) nas tabelas Delta do bucket `bronze` e demonstra
o **histórico de versões** e o **Time Travel** do Delta Lake.

```
MinIO / bronze / <tabela>  →  INSERT / UPDATE / DELETE  →  HISTORY / TIME TRAVEL
```

> Execute o notebook `02_csv_to_delta.ipynb` antes deste.


In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"]           = "C:\\hadoop"
os.environ["JAVA_TOOL_OPTIONS"]     = "-Djava.library.path=C:\\hadoop\\bin"

MINIO_ENDPOINT = "http://localhost:9020"
MINIO_ACCESS   = "minioadmin"
MINIO_SECRET   = "minioadmin"
BRONZE         = "s3a://bronze"

In [ ]:
builder = (
    SparkSession.builder
    .appName("DML Delta Lake - MinIO")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",              MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",            MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",            MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access",     "true")
    .config("spark.hadoop.fs.s3a.impl",                  "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "2g")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark {spark.version} iniciado.")

## Estado inicial das tabelas


In [ ]:
for tabela in ["clientes", "produtos", "pedidos"]:
    print(f"\n=== {tabela.upper()} (estado inicial) ===")
    spark.read.format("delta").load(f"{BRONZE}/{tabela}").orderBy("id").show(truncate=False)

## INSERT — Novos registros

Insere um novo cliente, dois novos produtos e um novo pedido.


In [ ]:
# INSERT - novo cliente
spark.sql(f"""
    INSERT INTO delta.`{BRONZE}/clientes`
    VALUES (6, 'Fernanda Torres', 'fernanda@email.com', 'Tubarao')
""")

# INSERT - novos produtos
spark.sql(f"""
    INSERT INTO delta.`{BRONZE}/produtos`
    VALUES
        (6, 'Webcam HD', 'Perifericos', 180.0, 60),
        (7, 'SSD 1TB',   'Eletronicos', 450.0, 40)
""")

# INSERT - novo pedido
spark.sql(f"""
    INSERT INTO delta.`{BRONZE}/pedidos`
    VALUES (6, 1, 6, 1, 180.0, '2024-01-25', 'processando')
""")

print("\n=== CLIENTES após INSERT ===")
spark.read.format("delta").load(f"{BRONZE}/clientes").orderBy("id").show(truncate=False)

print("\n=== PRODUTOS após INSERT ===")
spark.read.format("delta").load(f"{BRONZE}/produtos").orderBy("id").show(truncate=False)

print("\n=== PEDIDOS após INSERT ===")
spark.read.format("delta").load(f"{BRONZE}/pedidos").orderBy("id").show(truncate=False)

## UPDATE — Atualização de dados

Reajusta em 10% o preço dos produtos da categoria **Eletronicos** e marca os pedidos
com status `entregue` como `finalizado`.


In [ ]:
# UPDATE - reajuste de 10% nos Eletronicos
dt_produtos = DeltaTable.forPath(spark, f"{BRONZE}/produtos")
dt_produtos.update(
    condition="categoria = 'Eletronicos'",
    set={"preco": "preco * 1.10"},
)

# UPDATE - pedidos entregues → finalizado
dt_pedidos = DeltaTable.forPath(spark, f"{BRONZE}/pedidos")
dt_pedidos.update(
    condition="status = 'entregue'",
    set={"status": "'finalizado'"},
)

print("\n=== PRODUTOS após UPDATE (Eletronicos +10%) ===")
spark.read.format("delta").load(f"{BRONZE}/produtos").orderBy("id").show(truncate=False)

print("\n=== PEDIDOS após UPDATE (entregue → finalizado) ===")
spark.read.format("delta").load(f"{BRONZE}/pedidos").orderBy("id").show(truncate=False)

## DELETE — Remoção de registros

Remove os produtos com estoque zerado.


In [ ]:
# Zera o estoque do produto 5 para demonstrar o DELETE
dt_produtos.update(condition="id = 5", set={"estoque": "0"})

# DELETE - remove produtos sem estoque
dt_produtos.delete(condition="estoque = 0")

print("\n=== PRODUTOS após DELETE (sem estoque) ===")
spark.read.format("delta").load(f"{BRONZE}/produtos").orderBy("id").show(truncate=False)

## DESCRIBE HISTORY — Histórico de versões

O Delta Lake registra cada operação como uma nova versão da tabela.


In [ ]:
print("\n=== HISTÓRICO: produtos ===")
dt_produtos.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

print("\n=== HISTÓRICO: pedidos ===")
dt_pedidos.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## TIME TRAVEL — Leitura de versões anteriores

Com Time Travel é possível consultar o estado da tabela em qualquer versão anterior.


In [ ]:
# Versão 0 = estado original após a carga do CSV (notebook 02)
print("\n=== PRODUTOS - versão 0 (estado original) ===")
spark.read.format("delta").option("versionAsOf", 0).load(f"{BRONZE}/produtos").orderBy("id").show(truncate=False)

# Versão atual
print("\n=== PRODUTOS - versão atual ===")
spark.read.format("delta").load(f"{BRONZE}/produtos").orderBy("id").show(truncate=False)

In [ ]:
spark.stop()
print("Sessão Spark encerrada.")